# LangChain RAG Example (FAISS + OpenAI)

This notebook demonstrates:
- Loading documents (PDF/TXT)
- Splitting into chunks
- Creating embeddings
- Storing in FAISS
- Querying using retrieved context


In [1]:
#!pip install -U langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf

In [2]:
import os
from pathlib import Path
from pprint import pprint

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

from dotenv import load_dotenv



In [3]:
# ---------------------------------------------------------------------------
# Load environment variables from a .env file
# ---------------------------------------------------------------------------
load_dotenv()



#os.environ['OPENAI_API_KEY'] = 'your-api-key'

True

In [4]:
DOCS_DIR = 'docs'
VECTOR_DIR = 'faiss_index'

def load_documents(folder):
    docs = []
    for file_path in Path(folder).glob('*'):
        if file_path.suffix.lower() == '.pdf':
            loader = PyPDFLoader(str(file_path))
            print(f"Loading {file_path.name}")
            docs.extend(loader.load())
        elif file_path.suffix.lower() in ['.txt', '.md']:
            loader = TextLoader(str(file_path), encoding='utf-8')
            docs.extend(loader.load())
    return docs

In [5]:
def build_vector_store():
    documents = load_documents(DOCS_DIR)

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local(VECTOR_DIR)

    print(f'Saved {len(chunks)} chunks to {VECTOR_DIR}')

In [ ]:
def ask_question(question):
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

    vector_store = FAISS.load_local(VECTOR_DIR, embeddings, allow_dangerous_deserialization=True)

    retriever = vector_store.as_retriever(
        search_type='mmr',
        search_kwargs={'k': 8, 'fetch_k': 20}
        )
    docs = retriever.invoke(question)

    context = '\n\n'.join(doc.page_content for doc in docs)

    prompt = ChatPromptTemplate.from_template('''
You are a helpful assistant. Answer using only the context.
If unknown, say you do not know.

Context:
{context}

Question:
{question}
''')

    llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

    chain = prompt | llm

    response = chain.invoke({'context': context, 'question': question})

    return response.content

In [7]:
# Step 1: Put files in ./docs folder
# Step 2: Run this

build_vector_store()

Loading NLP_book_Jan25.pdf
Saved 2420 chunks to faiss_index


In [8]:
# Ask a question

answer = ask_question('Do a comparison of n-gram language models versus LLMs')
pprint(answer)

('Based on the provided context, here is a comparison of n-gram language '
 'models versus large language models (LLMs):\n'
 '\n'
 '1. **Model Basis:**\n'
 '   - **N-gram models:** Based on counting occurrences of sequences of words '
 '(n-grams) and estimating probabilities from these counts.\n'
 '   - **LLMs:** Based on neural networks rather than simple n-gram counts.\n'
 '\n'
 '2. **Parameter Growth:**\n'
 '   - **N-gram models:** The number of parameters increases exponentially as '
 'the n-gram order increases, making higher-order n-grams computationally '
 'expensive.\n'
 '   - **LLMs:** Neural networks can handle large parameter spaces more '
 'efficiently and do not suffer from exponential parameter growth in the same '
 'way.\n'
 '\n'
 '3. **Generalization:**\n'
 '   - **N-gram models:** Have no way to generalize from training examples to '
 'unseen test examples unless they use additional techniques like caches or '
 'class-based models, which provide only minor improvements